In [1]:
import pandas as pd
import os

print("Building the Business Master Dataset (Hourly Grain)...")

# Set your directories
base_path = r"E:\Labmentix Internship\STRAVA"
export_dir = r"C:\Users\tatha\Downloads\New folder"
export_path = os.path.join(export_dir, "Fitbit_Business_Master_Hourly.csv")

# ==========================================
# 1. LOAD & AGGREGATE MINUTE/SECOND DATA TO HOURLY
# ==========================================
print("Aggregating granular data to the Hourly level...")

hr = pd.read_csv(os.path.join(base_path, "heartrate_seconds_merged.csv"))
hr['Datetime_Hour'] = pd.to_datetime(hr['Time']).dt.floor('H') 
hr_hourly = hr.groupby(['Id', 'Datetime_Hour'], as_index=False)['Value'].mean().rename(columns={'Value': 'Avg_HR'})

mets = pd.read_csv(os.path.join(base_path, "minuteMETsNarrow_merged.csv"))
mets['Datetime_Hour'] = pd.to_datetime(mets['ActivityMinute']).dt.floor('H')
mets_hourly = mets.groupby(['Id', 'Datetime_Hour'], as_index=False)['METs'].mean().rename(columns={'METs': 'Avg_METs'})

# ==========================================
# 2. LOAD & RENAME HOURLY DATA (FIX APPLIED HERE)
# ==========================================
print("Loading native hourly data and renaming to prevent collisions...")
hc = pd.read_csv(os.path.join(base_path, "hourlyCalories_merged.csv"))
hc['Datetime_Hour'] = pd.to_datetime(hc['ActivityHour'])
hc.rename(columns={'Calories': 'HourlyCalories'}, inplace=True)  # Renamed
hc.drop(columns=['ActivityHour'], inplace=True)

hs = pd.read_csv(os.path.join(base_path, "hourlySteps_merged.csv"))
hs['Datetime_Hour'] = pd.to_datetime(hs['ActivityHour'])
hs.rename(columns={'StepTotal': 'HourlySteps'}, inplace=True)    # Renamed
hs.drop(columns=['ActivityHour'], inplace=True)

hi = pd.read_csv(os.path.join(base_path, "hourlyIntensities_merged.csv"))
hi['Datetime_Hour'] = pd.to_datetime(hi['ActivityHour'])
hi.rename(columns={'TotalIntensity': 'HourlyTotalIntensity', 'AverageIntensity': 'HourlyAvgIntensity'}, inplace=True) # Renamed
hi.drop(columns=['ActivityHour'], inplace=True)

# ==========================================
# 3. LOAD NATIVE DAILY DATA 
# ==========================================
print("Loading daily data...")
da = pd.read_csv(os.path.join(base_path, "dailyActivity_merged.csv"))
da['Date'] = pd.to_datetime(da['ActivityDate']).dt.date
da.rename(columns={'Calories': 'DailyCalories'}, inplace=True) # Renamed to prevent collision
da.drop(columns=['ActivityDate'], inplace=True)

sd = pd.read_csv(os.path.join(base_path, "sleepDay_merged.csv"))
sd['Date'] = pd.to_datetime(sd['SleepDay']).dt.date
sd.drop(columns=['SleepDay'], inplace=True)

wl = pd.read_csv(os.path.join(base_path, "weightLogInfo_merged.csv"))
wl['Date'] = pd.to_datetime(wl['Date']).dt.date
wl.drop(columns=['Fat', 'LogId'], errors='ignore', inplace=True)

# ==========================================
# 4. MERGE EVERYTHING INTO A SINGLE TIMELINE
# ==========================================
print("Merging all datasets into a single timeline...")

master_df = pd.merge(hc, hs, on=['Id', 'Datetime_Hour'], how='outer')
master_df = pd.merge(master_df, hi, on=['Id', 'Datetime_Hour'], how='outer')
master_df = pd.merge(master_df, hr_hourly, on=['Id', 'Datetime_Hour'], how='outer')
master_df = pd.merge(master_df, mets_hourly, on=['Id', 'Datetime_Hour'], how='outer')

# Extract 'Date' from the Hour column to join the Daily metrics
master_df['Date'] = master_df['Datetime_Hour'].dt.date

# Join Daily metrics
master_df = pd.merge(master_df, da, on=['Id', 'Date'], how='left')
master_df = pd.merge(master_df, sd, on=['Id', 'Date'], how='left')
master_df = pd.merge(master_df, wl, on=['Id', 'Date'], how='left')

# ==========================================
# 5. SMART IMPUTATION (ELIMINATING BLANKS SAFELY)
# ==========================================
print("Applying smart imputations to eliminate blanks...")

# A. Hourly metrics safely filled with 0 using the newly renamed columns
activity_cols = ['HourlyCalories', 'HourlySteps', 'HourlyTotalIntensity', 'HourlyAvgIntensity', 'Avg_METs']
master_df[activity_cols] = master_df[activity_cols].fillna(0)

# B. Smart Heart Rate Fill
master_df['Avg_HR'] = master_df.groupby(['Id', 'Date'])['Avg_HR'].transform(lambda x: x.fillna(x.mean()))
master_df['Avg_HR'] = master_df.groupby('Id')['Avg_HR'].transform(lambda x: x.fillna(x.mean()))
master_df['Avg_HR'] = master_df['Avg_HR'].fillna(70) 

# C. Forward Fill Weight & Sleep 
master_df.sort_values(['Id', 'Datetime_Hour'], inplace=True)
daily_cols = ['TotalSleepRecords', 'TotalMinutesAsleep', 'TotalTimeInBed', 'WeightKg', 'WeightPounds', 'BMI']
master_df[daily_cols] = master_df.groupby('Id')[daily_cols].ffill().bfill()

# D. Fill any remaining blanks with 0 safely
master_df.fillna(0, inplace=True)

# ==========================================
# 6. EXPORT
# ==========================================
master_df['Datetime_Hour'] = master_df['Datetime_Hour'].dt.strftime('%Y-%m-%d %H:%M:%S')
master_df.drop(columns=['Date'], inplace=True) 

os.makedirs(export_dir, exist_ok=True)
master_df.to_csv(export_path, index=False)

print(f"Success! Master dataset exported to: {export_path}")
print(f"Final Dataset Shape: {master_df.shape} (Rows, Columns)")

Building the Business Master Dataset (Hourly Grain)...
Aggregating granular data to the Hourly level...


C:\Users\tatha\AppData\Local\Temp\ipykernel_10436\1937229183.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hr['Datetime_Hour'] = pd.to_datetime(hr['Time']).dt.floor('H')
C:\Users\tatha\AppData\Local\Temp\ipykernel_10436\1937229183.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  mets['Datetime_Hour'] = pd.to_datetime(mets['ActivityMinute']).dt.floor('H')
C:\Users\tatha\AppData\Local\Temp\ipykernel_10436\1937229183.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  mets['Datetime_Hour'] = pd.to_datetime(mets['ActivityMinute']).dt.floor('H')


Loading native hourly data and renaming to prevent collisions...


C:\Users\tatha\AppData\Local\Temp\ipykernel_10436\1937229183.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  hc['Datetime_Hour'] = pd.to_datetime(hc['ActivityHour'])
C:\Users\tatha\AppData\Local\Temp\ipykernel_10436\1937229183.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  hs['Datetime_Hour'] = pd.to_datetime(hs['ActivityHour'])
C:\Users\tatha\AppData\Local\Temp\ipykernel_10436\1937229183.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  hi['Datetime_Hour'] = pd.to_datetime(hi['ActivityHour'])


Loading daily data...
Merging all datasets into a single timeline...


C:\Users\tatha\AppData\Local\Temp\ipykernel_10436\1937229183.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sd['Date'] = pd.to_datetime(sd['SleepDay']).dt.date
C:\Users\tatha\AppData\Local\Temp\ipykernel_10436\1937229183.py:57: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  wl['Date'] = pd.to_datetime(wl['Date']).dt.date


Applying smart imputations to eliminate blanks...
Success! Master dataset exported to: C:\Users\tatha\Downloads\New folder\Fitbit_Business_Master_Hourly.csv
Final Dataset Shape: (22178, 28) (Rows, Columns)
